# 실험 결과 비교, 평균 ± 표준편차, paired t-test

In [1]:
import os

import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt

T_LIST = [22, 30, 100, 300]
T_CSV = {T: f"exp_diffusion_step_{T}/exp_diffusion_step_{T}.csv" for T in T_LIST}
CLASS_NAMES = {"0": "Normal", "1": "Underfill_50FR", "2": "Underfill_Fan", "macro": "macro"}

frames = []
for T, path in T_CSV.items():
    if not os.path.exists(path):
        print("결과 CSV 없음, 건너뜀:", path); continue
    frames.append(pd.read_csv(path).assign(source=path, method=f"T={T}"))
df_all = pd.concat(frames, ignore_index=True)
df_all["class"] = df_all["class"].astype(str)
df_all = df_all.drop_duplicates(subset=["method", "seed", "class"], keep="last")

print("읽은 결과 파일:")
for p in df_all["source"].unique():
    print("  ", p)
print("\n실험별 seed 수:")
print(df_all[df_all["class"] == "macro"].groupby("method")["seed"].nunique().to_string())

읽은 결과 파일:
   exp_diffusion_step_22/exp_diffusion_step_22.csv
   exp_diffusion_step_30/exp_diffusion_step_30.csv
   exp_diffusion_step_100/exp_diffusion_step_100.csv
   exp_diffusion_step_300/exp_diffusion_step_300.csv

실험별 seed 수:
method
T=100    10
T=22     10
T=30     10
T=300    10


## 1. 실험별 평균 ± 표준편차

In [2]:
METHOD_ORDER = [f"T={T}" for T in T_LIST if f"T={T}" in set(df_all["method"])]


def per_seed(metric, cls="macro"):
    sub = df_all[df_all["class"] == str(cls)]
    return sub.pivot_table(index="seed", columns="method", values=metric, aggfunc="mean")[METHOD_ORDER]


summary_items = [("accuracy", "macro"), ("precision", "macro"), ("recall", "macro"), ("f1", "macro")]
summary = {}
for metric, cls in summary_items:
    tab = per_seed(metric, cls)
    name = f"{metric}" if cls == "macro" else f"{metric}_{CLASS_NAMES[cls]}"
    summary[name] = tab.mean().map("{:.3f}".format) + " ± " + tab.std(ddof=1).map("{:.3f}".format)
summary = pd.DataFrame(summary)
summary.insert(0, "n_seeds", per_seed("f1").notna().sum())
summary

,n_seeds,accuracy,precision,recall,f1
method,,,,,
T=22,10,0.872 ± 0.030,0.878 ± 0.028,0.872 ± 0.030,0.871 ± 0.030
T=30,10,0.884 ± 0.032,0.890 ± 0.028,0.884 ± 0.032,0.884 ± 0.034
T=100,10,0.896 ± 0.020,0.898 ± 0.020,0.896 ± 0.020,0.895 ± 0.020
T=300,10,0.894 ± 0.023,0.900 ± 0.019,0.894 ± 0.023,0.894 ± 0.024


## 2. paired t-test

In [3]:
COMPARISONS = [
    ("T=30", "T=22", "T30 vs T22"),
    ("T=100", "T=22", "T100 vs T22"),
    ("T=100", "T=30", "T100 vs T30"),
    ("T=300", "T=22", "T300 vs T22"),
    ("T=300", "T=30", "T300 vs T30"),
    ("T=300", "T=100", "T300 vs T100"),
]


def paired_tests(metric="f1", cls="macro", comparisons=COMPARISONS, df=None):
    df = df_all if df is None else df
    sub = df[df["class"] == str(cls)]
    table = sub.pivot_table(index="seed", columns="method", values=metric, aggfunc="mean")
    rows = []
    for a, b, desc in comparisons:
        if a not in table or b not in table:
            continue
        pair = table[[a, b]].dropna()
        n = len(pair)
        diff = pair[a] - pair[b]
        row = {"비교": desc, "A": a, "B": b, "n": n,
               "mean_A": pair[a].mean(), "mean_B": pair[b].mean()}
        if n >= 2 and diff.std(ddof=1) > 0:
            t_stat, p_val = stats.ttest_rel(pair[a], pair[b])
            row.update({"t": t_stat, "p": p_val})
        else:
            row.update({"t": np.nan, "p": np.nan})
        rows.append(row)
    res = pd.DataFrame(rows)
    if res.empty:
        print(f"[{metric}/{cls}] 비교할 수 있는 실험 쌍이 아직 없습니다.")
        return res
    res["유의(p<0.05)"] = res["p"] < 0.05
    return res


pd.set_option("display.float_format", "{:.4f}".format)
paired_tests("f1", "macro")


,비교,A,B,n,mean_A,mean_B,t,p,유의(p<0.05)
0,T30 vs T22,T=30,T=22,10,0.8836,0.8713,0.9255,0.3789,False
1,T100 vs T22,T=100,T=22,10,0.8952,0.8713,2.7725,0.0217,True
2,T100 vs T30,T=100,T=30,10,0.8952,0.8836,1.1865,0.2658,False
3,T300 vs T22,T=300,T=22,10,0.8938,0.8713,2.1038,0.0647,False
4,T300 vs T30,T=300,T=30,10,0.8938,0.8836,0.8158,0.4357,False
5,T300 vs T100,T=300,T=100,10,0.8938,0.8952,-0.1323,0.8976,False
